In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("harrimansaragih/dummy-advertising-and-sales-data")

print("Path to dataset files:", path)

/Users/prakashsah/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/prakashsah/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 92.7k/92.7k [00:00<00:00, 226kB/s]

Extracting files...
Path to dataset files: /Users/prakashsah/.cache/kagglehub/datasets/harrimansaragih/dummy-advertising-and-sales-data/versions/1


In [5]:
print(path
)

/Users/prakashsah/.cache/kagglehub/datasets/harrimansaragih/dummy-advertising-and-sales-data/versions/1


In [ ]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
)

with engine.connect() as conn:

    conn.execute(text("DROP TABLE IF EXISTS sales"))

    conn.commit()

print("sales table deleted successfully!")

sales table deleted successfully!


In [7]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
)

with engine.connect() as conn:

    result = conn.execute(text("SHOW TABLES"))

    tables = result.fetchall()

    print("\nDATABASE TABLES:\n")

    for table in tables:

        print(table[0])


DATABASE TABLES:



In [3]:
import pandas as pd
df=pd.read_csv("/Users/prakashsah/text2sql-agent/Dummy Data HSS.csv")

In [12]:
# -----------------------------------
# Total Sales by Influencer Type
# -----------------------------------
sales_by_influencer = (

    df.groupby("Influencer")["Sales"]
    .sum()
    .sort_values(ascending=False)

)

print(sales_by_influencer)

Influencer
Micro    220964.077708
Mega     220326.277952
Macro    219282.846469
Nano     218229.303090
Name: Sales, dtype: float64


In [13]:
# Generate 100K+ Relational Data And Insert Into MySQL

import pandas as pd
import numpy as np
import random
from faker import Faker

from sqlalchemy import create_engine, text

import os
from dotenv import load_dotenv


# =====================================================
# LOAD ENV
# =====================================================
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")


# =====================================================
# CREATE ENGINE
# =====================================================
engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
)


# =====================================================
# FAKER
# =====================================================
fake = Faker()


# =====================================================
# DELETE OLD TABLES
# =====================================================
with engine.connect() as conn:

    conn.execute(text("SET FOREIGN_KEY_CHECKS=0"))

    conn.execute(text("DROP TABLE IF EXISTS sales"))
    conn.execute(text("DROP TABLE IF EXISTS customers"))
    conn.execute(text("DROP TABLE IF EXISTS products"))

    conn.execute(text("SET FOREIGN_KEY_CHECKS=1"))

    conn.commit()

print("Old tables deleted")


# =====================================================
# CREATE TABLES
# =====================================================
create_customers = """
CREATE TABLE customers (

    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(255),
    customer_age INT,
    customer_gender VARCHAR(50),
    city VARCHAR(100),
    region VARCHAR(100),
    country VARCHAR(100),
    customer_segment VARCHAR(100)

);
"""


create_products = """
CREATE TABLE products (

    product_id INT PRIMARY KEY,
    product_name VARCHAR(255),
    category VARCHAR(100),
    sub_category VARCHAR(100),
    brand VARCHAR(100),
    unit_price FLOAT,
    cost_price FLOAT

);
"""


create_sales = """
CREATE TABLE sales (

    sale_id INT PRIMARY KEY,
    order_id VARCHAR(255),
    sale_date DATE,

    customer_id INT,
    product_id INT,

    quantity INT,
    discount_percent FLOAT,
    tax_percent FLOAT,
    shipping_cost FLOAT,

    payment_method VARCHAR(100),
    sales_channel VARCHAR(100),

    rating FLOAT,
    returned TINYINT,
    delivery_days INT,

    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id),

    FOREIGN KEY (product_id)
        REFERENCES products(product_id)

);
"""


with engine.connect() as conn:

    conn.execute(text(create_customers))
    conn.execute(text(create_products))
    conn.execute(text(create_sales))

    conn.commit()

print("Tables created")


# =====================================================
# GENERATE CUSTOMERS DATA (100K)
# =====================================================
customer_segments = [
    "Premium",
    "Regular",
    "Loyal",
    "Corporate"
]

regions = [
    "North",
    "South",
    "East",
    "West"
]

countries = [
    "USA",
    "India",
    "Nepal",
    "Canada"
]

customers = []

for i in range(1, 100001):

    customers.append({

        "customer_id": i,
        "customer_name": fake.name(),
        "customer_age": random.randint(18, 70),
        "customer_gender": random.choice([
            "Male",
            "Female"
        ]),
        "city": fake.city(),
        "region": random.choice(regions),
        "country": random.choice(countries),
        "customer_segment": random.choice(customer_segments)
    })


customers_df = pd.DataFrame(customers)

print("Customers generated")


# =====================================================
# GENERATE PRODUCTS DATA (100K)
# =====================================================

categories = {

    "Electronics": [
        "Laptop",
        "Phone",
        "Tablet"
    ],

    "Fashion": [
        "Shoes",
        "Jacket",
        "Watch"
    ],

    "Home": [
        "Chair",
        "Table",
        "Sofa"
    ]
}

brands = [
    "Apple",
    "Samsung",
    "Nike",
    "Rolex",
    "IKEA",
    "Sony"
]

products = []

for i in range(1, 100001):

    category = random.choice(list(categories.keys()))

    sub_category = random.choice(categories[category])

    unit_price = round(random.uniform(50, 5000), 2)

    cost_price = round(unit_price * random.uniform(0.4, 0.8), 2)

    products.append({

        "product_id": i,
        "product_name": f"{sub_category} {i}",
        "category": category,
        "sub_category": sub_category,
        "brand": random.choice(brands),
        "unit_price": unit_price,
        "cost_price": cost_price
    })


products_df = pd.DataFrame(products)

print("Products generated")


# =====================================================
# GENERATE SALES DATA (100K)
# =====================================================

payment_methods = [
    "Credit Card",
    "UPI",
    "Cash",
    "Debit Card"
]

sales_channels = [
    "Online",
    "Retail",
    "Mobile App"
]

sales = []

for i in range(1, 100001):

    sales.append({

        "sale_id": i,

        "order_id": f"ORD{i}",

        "sale_date": fake.date_between(
            start_date="-2y",
            end_date="today"
        ),

        "customer_id": random.randint(1, 100000),

        "product_id": random.randint(1, 100000),

        "quantity": random.randint(1, 10),

        "discount_percent": round(
            random.uniform(0, 30),
            2
        ),

        "tax_percent": round(
            random.uniform(1, 18),
            2
        ),

        "shipping_cost": round(
            random.uniform(5, 100),
            2
        ),

        "payment_method": random.choice(
            payment_methods
        ),

        "sales_channel": random.choice(
            sales_channels
        ),

        "rating": round(
            random.uniform(1, 5),
            1
        ),

        "returned": random.choice([
            0,
            1
        ]),

        "delivery_days": random.randint(1, 15)
    })


sales_df = pd.DataFrame(sales)

print("Sales generated")


# =====================================================
# INSERT INTO MYSQL
# =====================================================
customers_df.to_sql(
    name="customers",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

print("Customers inserted")


products_df.to_sql(
    name="products",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

print("Products inserted")


sales_df.to_sql(
    name="sales",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

print("Sales inserted")


# =====================================================
# VERIFY COUNTS
# =====================================================
with engine.connect() as conn:

    for table in [
        "customers",
        "products",
        "sales"
    ]:

        result = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )

        count = result.fetchone()[0]

        print(f"{table}: {count} rows")



Old tables deleted
Tables created
Customers generated
Products generated
Sales generated
Customers inserted
Products inserted
Sales inserted
customers: 100000 rows
products: 100000 rows
sales: 100000 rows


In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_google_google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

/Users/prakashsah/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/prakashsah/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm.invoke(
    "who is shahrukhan"
)

AIMessage(content='A question about the King of Bollywood. \n\nShah Rukh Khan, often referred to as SRK, is a highly acclaimed Indian film actor, producer, and television personality. He is widely regarded as one of the most successful and influential actors in the history of Indian cinema.\n\nBorn on November 2, 1965, in New Delhi, India, Shah Rukh Khan began his acting career in the late 1980s, initially working in television series and theater productions. His breakthrough role came in 1992 with the film "Deewana," which marked the beginning of his successful film career.\n\nWith a career spanning over three decades, Shah Rukh Khan has appeared in more than 80 films, including romantic dramas, comedies, and action movies. Some of his most notable films include:\n\n1. "Dilwale Dulhania Le Jayenge" (1995)\n2. "Kuch Kuch Hota Hai" (1998)\n3. "Devdas" (2002)\n4. "Chak De India" (2007)\n5. "My Name Is Khan" (2010)\n6. "Chennai Express" (2013)\n7. "Happy New Year" (2014)\n8. "Dilwale" (20